### Revisions code and analysis

## Mosaic the urban and cropland area 
The reclassifying happened in QGIS. According to the legend in [Data Download](https://storage.googleapis.com/earthenginepartners-hansen/GLCLU2000-2020/v2/download.html) as listed in the `Dataset Details`. The classifications that were kept were: 
1. Cropland (244 - 247) -> 1 
2. Cropland loss (248 - 249) -> 2
3. Stable build-up (250) -> 3
4. Gained build-up (251 - 253) -> 4

In [10]:
from pathlib import Path
import subprocess

#list of land cover files
folder_path = Path('/Volumes/student-drive/projects/ShallowGW/Land_Cover/processing_11-9-25')
vrt_path = folder_path / "merged.vrt"

tif_files = [str(tif) for tif in folder_path.glob('reclass_test_*.tif')]

subprocess.run(["gdalbuildvrt", str(vrt_path)] + [str(file) for file in tif_files])

0...10...20...30...40...50...60...70...80...90...100 - done.


CompletedProcess(args=['gdalbuildvrt', '/Volumes/student-drive/projects/ShallowGW/Land_Cover/processing_11-9-25/merged.vrt', '/Volumes/student-drive/projects/ShallowGW/Land_Cover/processing_11-9-25/reclass_test_40N-110W.tif', '/Volumes/student-drive/projects/ShallowGW/Land_Cover/processing_11-9-25/reclass_test_40N-120W.tif', '/Volumes/student-drive/projects/ShallowGW/Land_Cover/processing_11-9-25/reclass_test_50N-110W.tif', '/Volumes/student-drive/projects/ShallowGW/Land_Cover/processing_11-9-25/reclass_test_50N-120W.tif'], returncode=0)

Save the mosic file into the drive.

In [16]:
import rasterio as rio 
from rasterio.enums import Resampling

output_path = folder_path / 'merged_crop_urban.tif'

with rio.open(vrt_path) as src:
    profile = src.profile.copy()
    profile.update({
        "driver": "GTiff",
        "compress": "LZW",
        "tiled": True,
        "blockxsize": 512,
        "blockysize": 512
    })

    with rio.open(output_path, "w", **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window=window, resampling=Resampling.nearest)
            dst.write(data, window=window, indexes=1)

Crop the CRB for the mosaic data. 

In [30]:
# subprocess.run([
#     "gdalwarp",
#     "-cutline", "/Users/ppuente/github/detection_comparison-CRB/data/shapefiles/crb_boundary.shp",
#     "-crop_to_cutline",
#     "-dstnodata", "0",
#     "-co", "COMPRESS=LZW",
#     "-co", "TILED=YES",
#     str(output_path), #input file 
#     "/Users/ppuente/github/Surface_Water_Analysis-CRB/Data/processed-data/crb_cropland_urbanization_change2000-2020.tif" #output file
# ])

#new
subprocess.run([
    "gdalwarp",
    "-cutline", "/Users/ppuente/github/detection_comparison-CRB/data/shapefiles/crb_boundary.shp",
    "-crop_to_cutline",
    "-te", "-115.70625", "31.332", "-105.62625", "43.45225",  # xmin, ymin, xmax, ymax
    "-ts", "40320", "48481",  # target width and height
    "-dstnodata", "0",
    "-co", "COMPRESS=LZW",
    "-co", "TILED=YES",
    str(output_path),
    "/Users/ppuente/github/Surface_Water_Analysis-CRB/Data/processed-data/crb_cropland_urbanization_change2000-2020_new.tif"
])

Creating output file that is 40320P x 48481L.
Processing /Volumes/student-drive/projects/ShallowGW/Land_Cover/processing_11-9-25/merged_crop_urban.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.


CompletedProcess(args=['gdalwarp', '-cutline', '/Users/ppuente/github/detection_comparison-CRB/data/shapefiles/crb_boundary.shp', '-crop_to_cutline', '-te', '-115.70625', '31.332', '-105.62625', '43.45225', '-ts', '40320', '48481', '-dstnodata', '0', '-co', 'COMPRESS=LZW', '-co', 'TILED=YES', '/Volumes/student-drive/projects/ShallowGW/Land_Cover/processing_11-9-25/merged_crop_urban.tif', '/Users/ppuente/github/Surface_Water_Analysis-CRB/Data/processed-data/crb_cropland_urbanization_change2000-2020_new.tif'], returncode=0)

## Re-run urbanization analysis to include crop land and change the transitions to comparing 2000 to 2020

According to the land cover paper, they calculate the changes from 2000 to 2020 as a single year. So therefore we should calculate the transition changes as this too.

This code is Part 4 in the `analysis.ipynb` code file in the `Codes` folder.

In [1]:
from utils.helper_functions import WaterTransitionAnalyzer
import geopandas as gpd 
from shapely.geometry import mapping

#initialize the analyzer
transition_analyzer = WaterTransitionAnalyzer()

## Load shapefile with hucs and raster mode files
huc4_path = '../Data/processed-data/shapefiles/CRB_HUC4.shp'
HUC4_shapefile = gpd.read_file(huc4_path)

#file paths for rasters 
crb_2000_path = '../Data/processed-data/yearly_water_history-CRB/2000_CRB.tif'
crb_2020_path = '../Data/processed-data/yearly_water_history-CRB/2020_CRB.tif'
land_use_path = '../Data/processed-data/crb_cropland_urbanization_change2000-2020_new.tif'

In [2]:
# load huc raster only once, this takes the shapefile and creates a raster to keep track of which HUC the transition is occurring
HUC_shapes = [(mapping(row.geometry), int(row['huc4'])) for _, row in HUC4_shapefile.iterrows()]
#load single raster file to know what the shapefile needs to match 
src, transform = transition_analyzer.load_raster(crb_2000_path)
#Rasterize the shapefile
HUC_raster = transition_analyzer.raster_shapes(HUC_shapes, src.shape, transform)

In [3]:
# Compute transitions
trans_urban_results_df = transition_analyzer.calculate_transition_urban(crb_2000_path,crb_2020_path,land_use_path,HUC_raster)

Processed HUC 1401
Processed HUC 1402
Processed HUC 1403
Processed HUC 1404
Processed HUC 1405
Processed HUC 1406
Processed HUC 1407
Processed HUC 1408
Processed HUC 1501
Processed HUC 1502
Processed HUC 1503
Processed HUC 1504
Processed HUC 1505
Processed HUC 1506
Processed HUC 1507


In [4]:
trans_urban_results_df

,huc4,HUC_count,gained_urban_area,stable_urban_area,crop_gain_area,crop_loss_area,dry_transition_area,wet_transition_area,stable_urban_dry_area,gained_urban_dry_area,stable_urban_wet_area,gained_urban_wet_area,crop_loss_dry_area,crop_gain_dry_area,crop_loss_wet_area,crop_gain_wet_area,urbanization_percent,cropland_change_perc
0,1401,42746322,476.3799,1841.2803,503.5077,47.0286,31.0572,18.4356,1.0512,2.0394,0.9549,0.8406,0.0387,0.1152,0.3303,0.0495,6.024326,1.431017
1,1402,34383242,332.4024,1060.5978,696.6927,46.2402,24.4107,7.7067,0.4824,1.3320,0.2934,0.3375,0.0216,0.2943,0.2592,0.0279,4.501548,2.400824
2,1403,35722783,120.0744,592.7949,182.0403,21.8169,21.8268,4.2588,0.1962,0.8271,0.0756,0.0558,0.1827,0.4077,0.0270,0.1143,2.217288,0.634072
3,1404,93245179,235.3023,1030.1031,235.8432,35.4015,39.8835,55.9674,0.2304,0.5139,0.4302,0.3798,0.6939,0.0900,1.0269,0.0234,1.507859,0.323216
4,1405,58342496,259.8066,958.7106,445.2966,65.7081,21.0546,17.3421,0.3159,0.3726,0.2268,0.2754,0.0648,0.2997,0.1998,0.0639,2.320621,0.973189
5,1406,63352074,301.2696,933.1101,1207.1664,76.4613,43.5681,28.7640,0.1719,0.4752,0.0765,0.1512,0.7209,1.3473,0.7866,0.6840,2.164938,2.251312
6,1407,57685958,60.9786,317.2329,155.0655,11.2995,265.0950,3.4542,0.0450,0.3393,0.0234,0.0108,1.0791,0.4428,0.0846,0.0000,0.728488,0.320442
7,1408,104328808,539.9775,2219.7420,1322.7714,156.5055,100.6263,26.6022,0.9936,1.9890,0.4716,0.3969,0.3186,3.1005,0.0774,0.0387,2.939126,1.575443
8,1501,126496342,848.4570,2577.7134,302.5224,29.8728,378.1062,23.1048,0.3375,2.0709,0.1998,0.3672,1.7748,2.0043,0.2493,0.1431,3.009459,0.291967
9,1502,111044747,400.7601,1560.4272,38.9997,26.7678,52.5024,47.7702,0.2970,0.3645,0.1953,0.1278,0.0747,0.9234,0.0891,0.4743,1.962359,0.065807


In [5]:
#get the percentages of each 
trans_urban_results_df['stable-urban_dry_perc'] = (trans_urban_results_df['stable_urban_dry_area'] / trans_urban_results_df['dry_transition_area'])*100
trans_urban_results_df['gained-urban_dry_perc'] = (trans_urban_results_df['gained_urban_dry_area'] / trans_urban_results_df['dry_transition_area'])*100

trans_urban_results_df['urban_dry_perc'] = trans_urban_results_df['stable-urban_dry_perc'] + trans_urban_results_df['gained-urban_dry_perc']

trans_urban_results_df['stable-urban_wet_perc'] = (trans_urban_results_df['stable_urban_wet_area'] / trans_urban_results_df['wet_transition_area'])*100
trans_urban_results_df['gained-urban_wet_perc'] = (trans_urban_results_df['gained_urban_wet_area'] / trans_urban_results_df['wet_transition_area'])*100

trans_urban_results_df['urban_wet_perc'] = trans_urban_results_df['stable-urban_wet_perc'] + trans_urban_results_df['gained-urban_wet_perc']

#get the percentages of each 
trans_urban_results_df['crop-loss_dry_perc'] = (trans_urban_results_df['crop_loss_dry_area'] / trans_urban_results_df['dry_transition_area'])*100
trans_urban_results_df['crop-gain_dry_perc'] = (trans_urban_results_df['crop_gain_dry_area'] / trans_urban_results_df['dry_transition_area'])*100

trans_urban_results_df['crop_dry_perc'] = trans_urban_results_df['crop-loss_dry_perc'] + trans_urban_results_df['crop-gain_dry_perc']

trans_urban_results_df['crop-loss_wet_perc'] = (trans_urban_results_df['crop_loss_wet_area'] / trans_urban_results_df['wet_transition_area'])*100
trans_urban_results_df['crop-gain_wet_perc'] = (trans_urban_results_df['crop_gain_wet_area'] / trans_urban_results_df['wet_transition_area'])*100

trans_urban_results_df['crop_wet_perc'] = trans_urban_results_df['crop-loss_wet_perc'] + trans_urban_results_df['crop-gain_wet_perc']

In [6]:
trans_urban_results_df

,huc4,HUC_count,gained_urban_area,stable_urban_area,crop_gain_area,crop_loss_area,dry_transition_area,wet_transition_area,stable_urban_dry_area,gained_urban_dry_area,...,urban_dry_perc,stable-urban_wet_perc,gained-urban_wet_perc,urban_wet_perc,crop-loss_dry_perc,crop-gain_dry_perc,crop_dry_perc,crop-loss_wet_perc,crop-gain_wet_perc,crop_wet_perc
0,1401,42746322,476.3799,1841.2803,503.5077,47.0286,31.0572,18.4356,1.0512,2.0394,...,9.951316,5.179652,4.559656,9.739309,0.124609,0.370928,0.495537,1.791642,0.268502,2.060145
1,1402,34383242,332.4024,1060.5978,696.6927,46.2402,24.4107,7.7067,0.4824,1.3320,...,7.432806,3.807077,4.379306,8.186383,0.088486,1.205619,1.294105,3.363307,0.362023,3.725330
2,1403,35722783,120.0744,592.7949,182.0403,21.8169,21.8268,4.2588,0.1962,0.8271,...,4.688273,1.775148,1.310228,3.085376,0.837044,1.867887,2.704932,0.633981,2.683855,3.317836
3,1404,93245179,235.3023,1030.1031,235.8432,35.4015,39.8835,55.9674,0.2304,0.5139,...,1.866185,0.768662,0.678609,1.447271,1.739817,0.225657,1.965474,1.834818,0.041810,1.876628
4,1405,58342496,259.8066,958.7106,445.2966,65.7081,21.0546,17.3421,0.3159,0.3726,...,3.270069,1.307800,1.588043,2.895843,0.307771,1.423442,1.731213,1.152110,0.368467,1.520577
5,1406,63352074,301.2696,933.1101,1207.1664,76.4613,43.5681,28.7640,0.1719,0.4752,...,1.485261,0.265957,0.525657,0.791615,1.654651,3.092400,4.747051,2.734668,2.377972,5.112641
6,1407,57685958,60.9786,317.2329,155.0655,11.2995,265.0950,3.4542,0.0450,0.3393,...,0.144967,0.677436,0.312663,0.990099,0.407062,0.167034,0.574096,2.449192,0.000000,2.449192
7,1408,104328808,539.9775,2219.7420,1322.7714,156.5055,100.6263,26.6022,0.9936,1.9890,...,2.964036,1.772786,1.491982,3.264768,0.316617,3.081202,3.397819,0.290953,0.145477,0.436430
8,1501,126496342,848.4570,2577.7134,302.5224,29.8728,378.1062,23.1048,0.3375,2.0709,...,0.636964,0.864755,1.589280,2.454036,0.469392,0.530089,0.999481,1.078997,0.619352,1.698348
9,1502,111044747,400.7601,1560.4272,38.9997,26.7678,52.5024,47.7702,0.2970,0.3645,...,1.259942,0.408832,0.267531,0.676363,0.142279,1.758777,1.901056,0.186518,0.992878,1.179396


In [7]:
#save results 
trans_urban_results_df.to_csv('../Data/analysis-data/transitions_urban_cropland_2000_2020.csv', index=False)